# Head-to-head @ 0.95 — legacy vs current BaCP (magnitude)

One sparsity, one criterion, one seed, same dense checkpoint, same honest eval. Everything that differs is listed:

| | legacy (submission design) | current (fixed) |
|---|---|---|
| contrastive loss | 2B×2B SupCon+NTXent over `cat([s,t])` | rectangular CAP Eq.1, B×N |
| projection heads | per-model, student's trainable | one frozen head, shared |
| tau | 0.07 | 0.15 |
| regime | 5 ep + 10 interleaved recovery | 60 ep continuous, cubic ramp 80%, recover 20% |
| classifier head | dense | pruned |

The **current** arm runs here; the **legacy** arm is produced by the chain job (task `legacy_replication`) and is read below when present. Records: `...s0.95.magnitude.seed1` and `...seed1.legacy`.

In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
for cand in [here, *here.parents]:
    if (cand / 'nb_common.py').exists():
        sys.path.insert(0, str(cand)); break
    if (cand / 'project' / 'test_notebooks' / 'nb_common.py').exists():
        sys.path.insert(0, str(cand / 'project' / 'test_notebooks')); break
else:
    raise RuntimeError('cannot find nb_common.py -- start the kernel inside the repo')
import nb_common as nb
info = nb.setup()

## Current config — BaCP + magnitude @ 0.95

Straight from `FAMILIES` (no overrides). Skips if its record exists.

In [ ]:
cell = nb.make_cell('resnet50', 'bacp', seed=1, pruner='magnitude', sparsity=0.95)
print({k: cell['config'][k] for k in
       ('contrastive_mode', 'proj_mode', 'tau', 'epochs', 'recovery_epochs',
        'prune_task_head') if k in cell['config']})
nb.run(cell, gpu=0)

## Comparison

Re-run this cell any time; it reads whatever has landed.

In [ ]:
import json, glob, os
root = os.environ['BACP_RESULTS_DIR']
want = {'current': 'static.bacp.resnet50.cifar10.s0.95.magnitude.seed1',
        'legacy':  'static.bacp.resnet50.cifar10.s0.95.magnitude.seed1.legacy'}
got = {}
for f in glob.glob(os.path.join(root, 'runs', '*.json')):
    rec = json.load(open(f, encoding='utf-8'))
    for label, key in want.items():
        if rec.get('experiment_group') == key:
            got[label] = rec

print(f'{"arm":<10} {"test acc":<10} {"sparsity":<10} {"tau":<6} {"loss":<8} {"proj":<14} {"epochs"}')
for label in ('legacy', 'current'):
    r = got.get(label)
    if not r:
        print(f'{label:<10} (not yet on disk)'); continue
    print(f'{label:<10} {r.get("test_acc_pct", 0):<10.2f} '
          f'{r.get("sparsity_reported", 0):<10.4f} {r.get("tau", "-")!s:<6} '
          f'{r.get("contrastive_mode", "-")!s:<8} {r.get("proj_mode", "-")!s:<14} '
          f'{r.get("epochs", "-")}')
print()
print('reference  I.P. magnitude 0.95 (fixed, head-pruned) = 91.42')
print('reference  submission BaCP+magnitude 0.95           = 93.58')
if len(got) == 2:
    d = got['legacy']['test_acc_pct'] - got['current']['test_acc_pct']
    print()
    print(f'legacy - current = {d:+.2f}')